# Chapter 3 Figures

Generate F01–F09 from the saved Chapter 3 tables and existing cached classifier/SemEval artifacts. This notebook does not fit classifiers, run LLM inference, or prepare features. Available figures are saved as 300-dpi PNG and SVG under the report run's `figures/` directory; unavailable configurations are recorded in `figure_inventory.csv`.

In [ ]:
import json
import os
import numpy as np
from itertools import product
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm
from notebooks.utils.chapter3_figures_utils import (
    build_external_document_scores, build_semeval_profile_rows, load_report_tables,
    plot_f01_window_transfer, plot_f02_confusion_matrices, plot_f03_document_tradeoff,
    plot_f04_document_length, plot_f05_external_score_distributions,
    plot_f06_name_that_llm_source_recall, plot_f07_semeval_profile,
    build_semeval_local_error_inputs, build_semeval_threshold_sweep_inputs,
    plot_f09_boundary_threshold_sweep, plot_f10_semeval_local_errors,
    plot_f08_semeval_failure, select_semeval_profile_examples, semeval_boundary_threshold_sweep,
    semeval_token_error_metrics, semeval_window_error_metrics,
    build_semeval_window_boundary_comparison,
    validate_external_score_counts,
)

DATA_ROOT = Path(os.environ.get('TEXT_DETECTION_DATA', 'D:/Projects/master_diploma/text-detection-data'))
RUN_ID = 'feature-vector-classification-2026-09-25-logreg-random-forest-max600-min100'
REPORT_RUN_ID = os.environ.get('CHAPTER3_REPORT_RUN_ID', 'chapter3-restructured-2026-10-27')
REPORT_ROOT = DATA_ROOT / 'experiment-results' / 'report' / 'chapter3-restructured' / REPORT_RUN_ID
FIGURE_ROOT = REPORT_ROOT / 'figures'
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
tables = load_report_tables(REPORT_ROOT)
figure_inventory_rows = []
SCORER_LLM = "gemma_3_270m_it"

SCORER_LLM = "littlelamb"
def record_figures(figure_id, paths, source, details=''):
    by_stem = {}
    for path in paths:
        by_stem.setdefault(path.stem, []).append(path.name)
    for stem, names in by_stem.items():
        figure_inventory_rows.append({
            'figure_id': figure_id, 'status': 'available', 'stem': stem,
            'files': '|'.join(sorted(names)), 'source': source, 'details': details, 'reason': '',
        })
    if not by_stem:
        figure_inventory_rows.append({
            'figure_id': figure_id, 'status': 'unavailable', 'stem': '', 'files': '',
            'source': source, 'details': details, 'reason': 'no supported data rows',
        })

## F01–F04 · Figures from saved report tables

In [ ]:
f01_paths = plot_f01_window_transfer(tables['T06_window_metrics_long'], FIGURE_ROOT)
record_figures('F01', f01_paths, 'tables/T06_window_metrics_long.csv')
f02_paths = plot_f02_confusion_matrices(tables['T06_window_metrics_long'], FIGURE_ROOT)
record_figures('F02', f02_paths, 'tables/T06_window_metrics_long.csv')
f03_paths = plot_f03_document_tradeoff(tables['T07_document_mean_window'], FIGURE_ROOT)
record_figures('F03', f03_paths, 'tables/T07_document_mean_window.csv')
f04_paths = plot_f04_document_length(tables['T08_document_length_metrics'], FIGURE_ROOT, minimum_class_support=20)
record_figures('F04', f04_paths, 'tables/T08_document_length_metrics.csv', 'only bins with at least 20 documents per class')
print(f'F01–F04 figure files: {len(f01_paths) + len(f02_paths) + len(f03_paths) + len(f04_paths)}')

## F05–F06 · Cached external document scores

Score only saved prepared features with stored classifiers. The Name That LLM cohort uses the same maximum of 1,800 common IDs and the 250-token minimum as notebook 06.

In [ ]:
external_document_scores, external_score_availability = build_external_document_scores(
    DATA_ROOT, run_id=RUN_ID, scorer_llms=('gemma_3_270m_it', 'littlelamb'),
        # DATA_ROOT, run_id=RUN_ID, scorer_llms=('gemma_3_270m_it', ),

    model_families=('logistic_regression', 'random_forest'), training_windows=(32, 64),
    evaluation_windows=(32, 64), n2llm_limit=1800, min_token_count=250,
)
validate_external_score_counts(external_document_scores, tables['T09_external_document_results'])
display(external_score_availability.groupby(['dataset_label', 'scorer_llm', 'status']).size().rename('configuration_count').reset_index())
f05_paths = plot_f05_external_score_distributions(external_document_scores, FIGURE_ROOT)
record_figures('F05', f05_paths, 'cached prepared windows + saved classifiers', 'document mean-window scores; threshold 0.5')
f06_paths = plot_f06_name_that_llm_source_recall(external_document_scores, FIGURE_ROOT)
record_figures('F06', f06_paths, 'cached Name That LLM windows + saved classifiers + source metadata', 'Gemma 270M/LittleLamb where available; >=250 tokens')
for _, missing in external_score_availability.loc[external_score_availability.status.eq('unavailable')].iterrows():
    affected_figures = ('F05', 'F06') if missing.dataset_label == 'Name That LLM' else ('F05',)
    for figure_id in affected_figures:
        figure_inventory_rows.append({
            'figure_id': figure_id, 'status': 'unavailable', 'stem': '', 'files': '',
            'source': 'cached prepared windows / saved classifiers',
            'details': json.dumps({key: missing[key] for key in ('dataset_label', 'scorer_llm', 'model_family', 'training_window', 'evaluation_window')}, default=str),
            'reason': str(missing.reason),
        })
print(f'F05–F06 figure files: {len(f05_paths) + len(f06_paths)}')

## F07–F08 · Deterministic SemEval examples

Use Gemma 3 270M with logistic regression and matched 32/32 plus 64/64 configurations. F07 selects the lowest finite boundary error; F08 selects the highest finite boundary error. Each selected text must exist for both window lengths.

In [ ]:
boundary_path = DATA_ROOT / 'experiment-results' / 'classification' / 'evaluations' / RUN_ID / 'semeval_boundary_predictions.csv'
boundary_columns = ['text_id', 'window_length', 'absolute_token_error', 'scorer_llm', 'feature_set', 'training_regime', 'training_window', 'model_family']
boundary_rows = pd.read_csv(boundary_path, usecols=boundary_columns)
boundary_rows = boundary_rows.loc[
    boundary_rows.scorer_llm.eq('gemma_3_270m_it')
    & boundary_rows.feature_set.eq('simple_similarity')
    & boundary_rows.training_regime.eq('Kaggle+MAGE+Outfox')
    & boundary_rows.model_family.eq('logistic_regression')
    & boundary_rows.training_window.eq(boundary_rows.window_length)
    & boundary_rows.window_length.isin([32, 64])
].copy()
common_boundary_ids = boundary_rows.groupby('text_id').window_length.nunique()
boundary_rows = boundary_rows.loc[boundary_rows.text_id.isin(common_boundary_ids[common_boundary_ids.eq(2)].index)]
semeval_profile_paths = []
for figure_id, correct in tqdm((('F07', True), ('F08', False)), desc='Selecting SemEval examples'):
    try:
        text_id, selection_rule = select_semeval_profile_examples(
            boundary_rows, correct=correct, required_window_lengths=(32, 64),
        )
    except ValueError as error:
        figure_inventory_rows.append({'figure_id': figure_id, 'status': 'unavailable', 'stem': '', 'files': '', 'source': str(boundary_path), 'details': '', 'reason': str(error)})
        continue
    try:
        profile_parts = []
        for training_window in tqdm((32, 64), desc=f'{figure_id} · scoring cached SemEval example', leave=False):
            profile_parts.append(build_semeval_profile_rows(
                DATA_ROOT, run_id=RUN_ID, scorer_llm='gemma_3_270m_it',
                model_family='logistic_regression', training_window=training_window,
                evaluation_window=training_window, text_id=text_id,
            ))
        profile_rows = pd.concat(profile_parts, ignore_index=True)
    except (FileNotFoundError, KeyError, ValueError) as error:
        figure_inventory_rows.append({'figure_id': figure_id, 'status': 'unavailable', 'stem': '', 'files': '', 'source': str(boundary_path), 'details': f'text_id={text_id}; {selection_rule}', 'reason': str(error)})
        continue
    if figure_id == 'F07':
        paths = plot_f07_semeval_profile(profile_rows, FIGURE_ROOT, text_id=text_id, selection_rule=selection_rule)
    else:
        paths = plot_f08_semeval_failure(profile_rows, FIGURE_ROOT, text_id=text_id, selection_rule=selection_rule)
    semeval_profile_paths.extend(paths)
    record_figures(figure_id, paths, 'cached SemEval prepared windows + saved classifier + boundary predictions', f'text_id={text_id}; {selection_rule}')
print(f'F07–F08 figure files: {len(semeval_profile_paths)}')

## F09 · SemEval boundary-threshold sensitivity

Sweep the MG-score threshold from 0 to 1 in increments of 0.01. At each threshold, predict the first boundary after three consecutive smoothed token scores meet/exceed the threshold, using the existing five-token median smoothing rule. If no sustained MG run is found, use token position 0. Plot mean absolute boundary error and no-crossing rate for matched 32/32 and 64/64 configurations.

In [ ]:
threshold_scored_windows, threshold_token_counts, threshold_gold_boundaries = build_semeval_threshold_sweep_inputs(
    DATA_ROOT, run_id=RUN_ID, scorer_llm='gemma_3_270m_it',
    model_family='logistic_regression', window_pairs=((32, 32), (64, 64)),
)
threshold_values = tuple(np.round(np.arange(0.0, 1.001, 0.01), 2))
boundary_threshold_metrics = semeval_boundary_threshold_sweep(
    threshold_scored_windows, threshold_token_counts, threshold_gold_boundaries,
    thresholds=threshold_values, smoothing_window=5, sustained_length=3,
)
boundary_threshold_metrics.to_csv(FIGURE_ROOT / 'F09_semeval_boundary_threshold_sweep.csv', index=False)
f09_paths = plot_f09_boundary_threshold_sweep(boundary_threshold_metrics, FIGURE_ROOT)
record_figures('F09', f09_paths, 'cached SemEval prepared windows + saved classifier + aligned gold boundaries', 'threshold 0..1 step 0.01; median smoothing=5; sustained_length=3')
display(boundary_threshold_metrics.groupby('window_length').agg(
    threshold_count=('threshold', 'nunique'), min_mae=('mean_absolute_token_error', 'min'),
    threshold_at_min_mae=('mean_absolute_token_error', lambda values: boundary_threshold_metrics.loc[values.index].loc[values.idxmin(), 'threshold']),
    min_no_crossing_rate=('no_crossing_rate', 'min'), document_count=('document_count', 'min'),
))
print(f'F09 figure files: {len(f09_paths)}')

## F10 · SemEval local HG/MG error and token-threshold sensitivity

Across every available saved SemEval scorer/classifier/window configuration, compare the fraction of pure HG and MG windows classified incorrectly at the existing 0.5 decision threshold. The token panel averages overlapping window MG-scores to token positions, includes boundary-crossing window scores, and shows the mislabeled-token fraction separately for gold HG and MG tokens over thresholds 0.00–1.00.

In [ ]:
(local_scored_windows, local_pure_windows, local_token_counts, local_gold_boundaries, local_configuration_availability) = build_semeval_local_error_inputs(
    DATA_ROOT, run_id=RUN_ID,
)
local_window_error_metrics = semeval_window_error_metrics(local_pure_windows)
local_token_thresholds = tuple(np.round(np.arange(0.0, 1.001, 0.01), 2))
local_token_error_metrics = semeval_token_error_metrics(
    local_scored_windows, local_token_counts, local_gold_boundaries, thresholds=local_token_thresholds,
)
local_window_error_metrics.to_csv(FIGURE_ROOT / 'F10_semeval_window_error_rates.csv', index=False)
local_token_error_metrics.to_csv(FIGURE_ROOT / 'F10_semeval_token_error_rates.csv', index=False)
f10_paths = plot_f10_semeval_local_errors(
    local_window_error_metrics, local_token_error_metrics, FIGURE_ROOT,
)
record_figures(
    'F10', f10_paths, 'cached SemEval prepared windows + saved classifiers + aligned gold boundaries',
    'pure windows at MG threshold 0.5; token thresholds 0.00..1.00 step 0.01',
)
for unavailable in local_configuration_availability.loc[local_configuration_availability.status.eq('unavailable')].itertuples(index=False):
    figure_inventory_rows.append({
        'figure_id': 'F10', 'status': 'unavailable', 'stem': '', 'files': '',
        'source': str(DATA_ROOT / 'experiment-results' / 'classification' / 'evaluations' / RUN_ID / 'semeval_summary.csv'),
        'details': f'{unavailable.scorer_llm}/{unavailable.model_family}/{unavailable.training_window}/{unavailable.evaluation_window}',
        'reason': unavailable.reason,
    })
display(local_configuration_availability.groupby(['scorer_llm', 'status']).size().rename('configuration_count').reset_index())
display(local_window_error_metrics.head())
print(f'F10 figure files: {len(f10_paths)}')

## T11 · Token-level vs window-run boundary MAE

Compare the existing token-projection boundary MAE with the 3-HG/3-MG window-run MAE on the same SemEval train documents and matched 32/32 and 64/64 configurations. If a method finds no boundary transition, score the prediction at token position 0; both no-transition rates are shown.

In [ ]:
boundary_prediction_path = DATA_ROOT / 'experiment-results' / 'classification' / 'evaluations' / RUN_ID / 'semeval_boundary_predictions.csv'
boundary_comparison_columns = [
    'text_id', 'window_length', 'token_count', 'gold_token_boundary', 'predicted_token_boundary',
    'absolute_token_error',
    'no_crossing', 'scorer_llm', 'feature_set', 'training_regime', 'training_window',
    'model_family', 'evaluation_dataset', 'semeval_split',
]
token_boundary_rows = pd.read_csv(boundary_prediction_path, usecols=boundary_comparison_columns)
window_boundary_comparison = build_semeval_window_boundary_comparison(
    local_scored_windows, token_boundary_rows, threshold=0.5, run_length=3,
)
t11_columns = [
    'scorer_llm', 'model_family', 'n_32', 'token_mae_32', 'window_mae_32',
    'token_no_crossing_pct_32', 'window_no_transition_pct_32', 'n_64',
    'token_mae_64', 'window_mae_64', 'token_no_crossing_pct_64',
    'window_no_transition_pct_64',
]
t11_comparison = window_boundary_comparison[t11_columns]
comparison_path = REPORT_ROOT / 'tables' / 'T11_semeval_window_boundary_comparison.csv'
comparison_path.parent.mkdir(parents=True, exist_ok=True)
t11_comparison.to_csv(comparison_path, index=False)
display(t11_comparison)
print(f'Window/token boundary comparison written to {comparison_path}')

## T12 · Window-boundary MAE when a transition is detected

This conditional MAE uses only documents with a detected 3-HG/3-MG transition. The detected count and percentage are shown so the conditional result is read alongside its coverage. No-transition documents are excluded here; their fallback-inclusive MAE remains in T11.

In [ ]:
t12_columns = [
    'scorer_llm', 'model_family', 'n_window_detected_32',
    'window_detected_mae_32', 'window_detection_pct_32',
    'n_window_detected_64', 'window_detected_mae_64', 'window_detection_pct_64',
]
detected_only_comparison = window_boundary_comparison[t12_columns].copy()
detected_only_path = REPORT_ROOT / 'tables' / 'T12_semeval_window_boundary_detected_only.csv'
detected_only_path.parent.mkdir(parents=True, exist_ok=True)
detected_only_comparison.to_csv(detected_only_path, index=False)
display(detected_only_comparison)
print(f'Detected-only window boundary MAE written to {detected_only_path}')

## Figure inventory

The inventory gives generated file pairs and the exact reason for each unavailable cached configuration.

In [ ]:
figure_inventory = pd.DataFrame(figure_inventory_rows)
inventory_path = FIGURE_ROOT / 'figure_inventory.csv'
figure_inventory.to_csv(inventory_path, index=False)
display(figure_inventory)
print(f'Figure inventory written to {inventory_path}')